# ⚖️ Cuaderno 02 — Eliminación Gaussiana

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es **taller, no libro**. La teoría completa —formulación matricial,
por qué las operaciones de renglón no cambian la solución, pivoteo parcial y
conteo de operaciones— está en el documento:

📖 [`docs/metodos/01_eliminacion_gaussiana.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md)

Aquí vas a **programar el método**, no a leerlo.

### 🎯 Lo que vas a entregar

- **Pregunta 9** — `mi_gauss_eliminar(A, b)`: recibe $\mathbf{A}$ y $\mathbf{b}$, devuelve la matriz aumentada ya **triangular**.
- **Pregunta 10** — `mi_sustitucion_atras(Ab)`: recibe la triangular, devuelve $\mathbf{x}$.
- **Pregunta 11** — `mi_gauss_solve(A, b)`: las dos anteriores encadenadas.

### ⚙️ Dos reglas del cuaderno

1. **Aritmética exacta.** Trabajamos con `M.sym`, no con decimales: verás $8/5$ y nunca `1.6000000000000003`. Así, si tu método falla, es un error de **método** y no de redondeo.
2. **Python empieza en 0.** El libro numera desde 1. En la sección 2 hay una tabla de equivalencias; consúltala cada vez que traduzcas un subíndice.


In [ ]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, subprocess, importlib

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # ============================================================
    # Google Colab: montar Drive y traer el repositorio
    # ============================================================
    # El repo se clona DENTRO de tu Drive, no en /content (que se borra al
    # cerrar la sesión). Así sobrevive entre sesiones y `git pull` solo
    # actualiza lo que cambió.
    from google.colab import drive

    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)

    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        # Lo más común: el alumno guardó el cuaderno DENTRO del clon (Colab lo
        # reescribe), así que git se niega a sobrescribirlo y el pull aborta.
        # Aquel alumno se quedaba sin la carpeta grader/, que es lo único
        # imprescindible. Aquí se fuerza sólo eso, sin tocar sus notebooks.
        print("⚠️ No pude actualizar el repositorio completo:", _exc)
        print("   Fuerzo solo lo que el cuaderno necesita (tus notebooks no se tocan)...")
        try:
            subprocess.run(["git", "-C", REPO, "fetch", "origin", "master"],
                           check=True)
            subprocess.run(["git", "-C", REPO, "checkout", "FETCH_HEAD", "--",
                            "grader", "lib/robomat.py"], check=True)
            print("✅ grader/ y lib/robomat.py quedaron al día.")
        except Exception as _exc2:
            print("⛔ Tampoco pude actualizar solo esos archivos:", _exc2)
            print(f"   Corre esto en una celda y vuelve a ejecutar la carga:")
            print(f"   !git -C '{REPO}' fetch origin master && git -C '{REPO}' checkout FETCH_HEAD -- grader lib/robomat.py")
else:
    # En tu computadora el repo ya está en disco: solo hay que encontrarlo.
    _ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# ============================================================
# Biblioteca del curso (lib/robomat.py)
# ============================================================
# En sys.path van las DOS raíces que hacen falta:
#   REPO/lib    -> robomat.py y el paquete mn_motor/
#   REPO        -> el paquete preguntas/ (las preguntas de este cuaderno)
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print("✅ Entorno cargado desde:", _robomat.__file__)


# ============================================================
# Motor de calificación
# ============================================================
# En Colab SIEMPRE se usa el blob ofuscado. En tu computadora puedes escoger
# "modulos": importa lib/mn_motor/ tal cual, y así funcionan F12, el
# autocompletado y los puntos de ruptura. Es el modo para depurar.
MODO_MOTOR = "modulos"

MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        # Se borran del caché para que una edición en lib/mn_motor/ o en
        # preguntas/ se note al volver a correr la celda, sin reiniciar.
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        # Import ESTÁTICO después del borrado: es lo que Pylance analiza.
        import mn_motor as _motor
        from mn_motor import *        # noqa: F401,F403
        if "usar_cuaderno" not in globals():
            # Un clon a medias deja lib/mn_motor/ como carpeta vacía: el import
            # "funciona" pero no trae nada, y eso NO es un ImportError. Sin
            # esto, el cuaderno moría más abajo con un NameError críptico.
            raise ImportError("lib/mn_motor está incompleto (no trae usar_cuaderno)")
    except ImportError as _exc:
        # En un clon del alumno no existe preguntas/: no hay clave de
        # respuestas. Sin problema: el blob trae lo mismo.
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        print("   Uso el motor ofuscado del blob.")
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 02.
    ruta_motor = os.path.join(REPO, "grader", "u3_02_gaussiana.txt")
    if not os.path.exists(ruta_motor):
        raise FileNotFoundError(
            f"No encuentro {ruta_motor}: tu copia del repositorio está incompleta."
            f" Corre esto en una celda y vuelve a ejecutar:"
            f" !git -C '{REPO}' fetch origin master && git -C '{REPO}'"
            f" checkout FETCH_HEAD -- grader lib/robomat.py")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    # _REPO_U3 le dice al motor dónde está el repo: dentro del blob no existe
    # __file__, así que no puede deducirlo solo.
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas (carpeta preguntas/u3_02_gaussiana).
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_02_gaussiana", "U3_02_Eliminacion_Gaussiana")

print(f"✅ Motor de calificación cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")


## 0 · Tu identificación

Todo lo que sigue usa **tus** datos. Al ejecutar la celda de abajo, Colab te
pide permiso para leer tu correo; de ahí salen tu número de control y tu nombre.
Con eso se siembran los números de tus preguntas: **cada alumno recibe datos
distintos**.

Fuera de Colab no hay sesión de Google, así que el correo se escribe a mano en
la celda.


In [ ]:
#@title 👤 Identificación automática (correo institucional)
# ============================================================
# Uso el correo con el que montaste Drive: de ahí salen tu NC y tu nombre.
# ============================================================
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    # Modo local: aquí no hay sesión de Google, escribe tu correo.
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""                                 # Google solo lo da en Colab

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
else:
    from IPython.display import HTML, display as _display

    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnito.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

# La tarea se genera UNA sola vez: todas las celdas de preguntas llaman a
# pregunta(n) sobre este MI_TAREA, así que todas hablan de lo mismo.
MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


## 1 · La idea en tres pasos

Resolver $\mathbf{A}\mathbf{x}=\mathbf{b}$ por eliminación es **repetir la misma
operación de renglón** hasta que la matriz quede triangular:

$$\text{original} \;\longrightarrow\; \text{triangular superior} \;\longrightarrow\; \text{despejar de abajo hacia arriba}$$

La primera flecha es la **eliminación hacia adelante**
([sección 1.2](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md#sec-1-2)
del documento) y la segunda la **sustitución hacia atrás**
([sección 1.4](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md#sec-1-4)).

El sistema que usaremos todo el cuaderno es:

$$2x_1 + x_2 - x_3 = 8, \qquad -3x_1 - x_2 + 2x_3 = -11, \qquad -2x_1 + x_2 + 2x_3 = -3$$

cuya solución es $x_1=2$, $x_2=3$, $x_3=-1$. Compruébala al final: si tu método
devuelve eso, va bien.

### Por qué la operación de renglón es legal

Restar un múltiplo de un renglón a otro **no cambia** el conjunto solución: es
exactamente lo que harías despejando a mano. Lo único que buscamos es que
aparezcan ceros debajo de la diagonal, y la cuenta es siempre la misma:

$$R_i \;\leftarrow\; R_i - m_{ik}R_k, \qquad m_{ik}=\frac{a_{ik}^{(k)}}{a_{kk}^{(k)}}$$

El número $m_{ik}$ se llama **multiplicador** y es justo el que hace cero la
posición $(i,k)$, porque $a_{ik}^{(k)} - m_{ik}\,a_{kk}^{(k)} = 0$.

> 🔎 El superíndice $(k)$ **no es una potencia**: es el número de paso. En el
> paso 1 el elemento vale una cosa y en el paso 2 puede valer otra, porque ya
> le restamos múltiplos del renglón pivote. Para el desarrollo completo, mira
> la [sección 1.1](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md#sec-1-1)
> del documento.


In [ ]:
#@title El sistema del cuaderno, con aritmética exacta
# M.sym construye la matriz con números EXACTOS (fracciones), no float64.
A = M.sym[[2, 1, -1],
          [-3, -1, 2],
          [-2, 1, 2]]
b = M.sym[[8],
          [-11],
          [-3]]

# Pegando A y b en un solo objeto sale la matriz aumentada [A | b].
Ab = M.sym[[A, b]]

mostrar(Ab)
print("tamaño:", Ab.shape, "· ¿exacto?", es_exacto(Ab))


## 2 · Pseudocódigo: eliminación hacia adelante

```
para k = 1, 2, ..., n-1:            # columna que se va a anular
    para i = k+1, ..., n:           # renglones debajo del pivote
        m = a[i][k] / a[k][k]
        para j = k, ..., n+1:       # columnas, incluida la de b
            a[i][j] = a[i][j] - m * a[k][j]
```

**Las tres cosas que hay que entender antes de teclearlo:**

1. El ciclo de `j` arranca en `k`, no en `1`. A la izquierda de la columna `k` ya hay ceros, y `0 - m·0 = 0`: recorrerlos es trabajo tirado.
2. Sí se toca la columna de $\mathbf{b}$ (que es la columna $n+1$ del pseudocódigo). Si no la actualizas, estás triangularizando $\mathbf{A}$ pero no el sistema.
3. Al terminar el paso `k`, la columna `k` ya tiene ceros debajo del pivote. El último renglón no se toca nunca, y por eso el ciclo externo para en `n-1`.

### Del libro a Python

El libro numera desde 1 y Python desde 0. **En el ciclo no hay que sumar ni
restar nada**: si `k` recorre `0, 1, …, n-2`, el pivote es `Ab[k, k]`,
exactamente como `a[k][k]` en el libro. Lo único que cambia es dónde
**termina** cada ciclo:

| | Libro | Python |
|---|---|---|
| renglones y columnas | $1\ldots n$ | `0 … n-1` |
| ciclo externo | `k = 1 … n-1` | `for k in range(n - 1)` |
| renglones debajo | `i = k+1 … n` | `for i in range(k + 1, n)` |
| columnas a restar | `j = k … n+1` | `for j in range(k, n + 1)` |

Otro detalle de la biblioteca: `Ab[i, :]` es un **renglón** de la matriz, así que
`Ab[i, :] = Ab[i, :] - m * Ab[k, :]` hace la resta de golpe, sin ciclo de `j`. Es
más corto, pero el ciclo deja ver mejor lo que pasa. Usa el que entiendas.


## 3 · Tu turno: eliminación hacia adelante

Escribe la función en la celda de abajo. `practica('prog_eliminar')` la prueba
con el sistema de la sección 1 y te dice si ya coincide; **no** te muestra la
respuesta, solo el veredicto. Puedes correrla cuantas veces quieras: la práctica
no gasta intentos.

Ojo con dos trampas que ya vimos:

- Devuelve la matriz **completa** (con la columna de $\mathbf{b}$), no solo $\mathbf{A}$.
- Tiene que servir para cualquier $n$. La pregunta 9 la va a probar con sistemas de **2, 3 y 4** incógnitas: si clavas los índices a un 3×3, los otros dos casos fallan.


In [ ]:
#@title ✍️ Escribe tu eliminación hacia adelante
def mi_gauss_eliminar(A, b):
    """Devuelve [A | b] triangularizada hacia adelante (sin pivoteo)."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 2")


# Cuando ya esté, esta línea te dice si coincide con el ejemplo.
practica('prog_eliminar')


In [ ]:
#@title 📝 Pregunta 9 (vale 1 punto)
pregunta(9)


## 4 · Sustitución hacia atrás

Con la matriz ya triangular, el sistema se resuelve de abajo hacia arriba: el
último renglón tiene **una sola** incógnita, el penúltimo dos, y así.

$$x_i = \frac{1}{a_{ii}}\left(b_i - \sum_{j=i+1}^{n} a_{ij}\,x_j\right), \qquad i = n, n-1, \ldots, 1$$

```
para i = n, n-1, ..., 1:
    suma = 0
    para j = i+1, ..., n:               # columnas a la derecha de la diagonal
        suma = suma + a[i][j] * x[j]
    x[i] = (a[i][n+1] - suma) / a[i][i]
```

**Lo que suele salir mal:** empezar por arriba. En el primer renglón están
**todas** las incógnitas, así que no hay nada que despejar todavía. Hay que
empezar por el último.

La suma va sobre las columnas **a la derecha** de la diagonal, porque esas son
las $x_j$ que ya conoces. Las de la izquierda valen cero y no aportan nada.

> 📐 La productoria $\prod$ que aparece en el documento para el determinante es
> al producto lo que $\sum$ es a la suma: multiplicar en vez de sumar. Y
> $u_{11}u_{22}\cdots u_{nn}$ (la diagonal de la triangular) es el resultado
> escrito a mano.


## 5 · Tu turno: sustitución hacia atrás

Ahora `mi_sustitucion_atras(Ab)` recibe una matriz **ya triangular** y devuelve
$\mathbf{x}$. La pregunta 10 también la prueba con tres tamaños distintos.

> 💡 Truco para el ciclo: `for i in range(n - 1, -1, -1)` recorre
> `n-1, n-2, …, 0`. El `-1` del final es *hasta antes de -1*, o sea, llega a `0`.


In [ ]:
#@title ✍️ Escribe tu sustitución hacia atrás
def mi_sustitucion_atras(Ab):
    """Devuelve x como columna, despejando de abajo hacia arriba."""
    raise NotImplementedError("Traduce aquí el pseudocódigo de la sección 4")


practica('prog_sustitucion')


In [ ]:
#@title 📝 Pregunta 10 (vale 1 punto)
pregunta(10)


## 6 · El método completo

Ya tienes las dos mitades. El método de eliminación gaussiana es, literalmente,
una llamada a la otra:

```
Gauss(A, b):
    Ab = eliminar_hacia_adelante(A, b)
    x  = sustituir_hacia_atras(Ab)
    devolver x
```

Escríbelo reusando **tus** funciones de arriba. Si las dos ya pasaron su
`practica`, esta debería pasar sola.


In [ ]:
#@title ✍️ Escribe el método completo
def mi_gauss_solve(A, b):
    """Resuelve A x = b y devuelve x como columna."""
    raise NotImplementedError("Encadena tus dos funciones")


practica('prog_solve')


In [ ]:
#@title 📝 Pregunta 11 (vale 1 punto)
pregunta(11)


## 7 · El pivoteo parcial, en una imagen

Todo lo anterior supone que el pivote $a_{kk}^{(k)}$ **no es cero**. Cuando lo es,
el método se atora: no hay por qué dividir entre cero. La solución es de sentido
común: **cambiar el orden de los renglones**, lo cual no altera el sistema.

El pivoteo parcial elige, entre los renglones que quedan, el de **mayor valor
absoluto** en la columna del pivote:

$$k^{*} = \arg\max_{i \ge k} \left| a_{ik}^{(k)} \right|$$

¿Por qué el de mayor valor absoluto y no cualquiera distinto de cero? Porque
dividir entre un número grande **amplifica menos** los errores de redondeo que
dividir entre uno diminuto. La demostración con $10^{-18}$ está en la
[sección 1.3](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md#sec-1-3)
del documento.

En este cuaderno no lo necesitas para las preguntas 9–11: los sistemas ocultos
se pueden eliminar sin pivoteo. Pero sí preguntan sobre él.


In [ ]:
#@title 🔀 Por qué hace falta el pivoteo
# Este sistema NO se puede eliminar sin intercambiar renglones: el pivote vale 0.
A0 = M.sym[[0, 2, 1], [1, 1, 1], [2, 1, 3]]
b0 = M.sym[[5], [4], [9]]
P = M.sym[[A0, b0]]
mostrar(P)

# Sin pivoteo, el multiplicador sería 1/0. SymPy no truena: devuelve "zoo"
# (infinito complejo) y todo lo que sigue queda contaminado.
print("m21 = 1/0 =", P[1, 0] / P[0, 0], " <- con esto no se puede seguir")

# Con pivoteo parcial: se sube al primer renglón el de mayor |a_i1|.
k = 0
for i in range(P.rows):
    if abs(P[i, 0]) > abs(P[k, 0]):
        k = i

filas = P.tolist()                 # el intercambio se hace sobre listas
filas[0], filas[k] = filas[k], filas[0]
Pp = M.sym[filas]

print(f"se intercambian los renglones 1 y {k + 1}")
mostrar(Pp)
print("el pivote ahora vale", Pp[0, 0], "y el método puede seguir")


## 8 · Preguntas de teoría (1 a 8)

Las primeras cinco son de opción múltiple; la 6 pide una matriz, la 7 y la 8 un
número. Todas se contestan con lo que ya trabajaste: el multiplicador, el
pivoteo, la forma triangular y el costo.


In [ ]:
#@title 🧠 Preguntas 1 a 5 · opción múltiple
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)
pregunta(5)


In [ ]:
#@title ✍️ Preguntas 6 a 8 · con tus matrices
# La 6 es una matriz (escríbela como [[1,2,3],[4,5,6],[7,8,9]]);
# la 7 y la 8 son un solo número.
pregunta(6)
pregunta(7)
pregunta(8)


## 9 · Calificar y enviar

- `calificar()` **no envía nada**: puedes usarla cuantas veces quieras mientras
  trabajas. Te dice qué preguntas están bien y, en las de programar, **qué caso**
  falló.
- `enviar(alumno_id)` sí manda la calificación y **gasta uno de tus 5 intentos**.
  Úsala cuando ya tengas la mayoría de los puntos.


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ ¿Qué sigue?

Ya tienes un resolvedor de sistemas lineales escrito por ti. En el **Cuaderno 03**
verás que la eliminación gaussiana guarda, sin querer, la información suficiente
para **volver a resolver** un sistema con el mismo $\mathbf{A}$ y distinto
$\mathbf{b}$: eso es la **descomposición LU**.

📖 Teoría de este cuaderno:
[`docs/metodos/01_eliminacion_gaussiana.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/master/docs/metodos/01_eliminacion_gaussiana.md)

📤 Antes de cerrar: ejecuta `calificar()` y luego `enviar(alumno_id)`. Recuerda
que tienes **5 intentos**.
